# Profiling Survey Kos Toba (928 Baris)

Notebook ini untuk eksplorasi 928 respons (`data/raw/Survey_Kos.csv`, 35 kolom) dan evaluasi 4 dimensi kualitas data.
Preprocessing produksi dijalankan oleh `etl/normalize_survey.py` agar dapat direproduksi.

In [1]:
from pathlib import Path
import pandas as pd

raw_path = Path('../data/raw/Survey_Kos.csv')
df = pd.read_csv(raw_path)
df.shape

In [2]:
quality = pd.DataFrame({
    'missing': df.isna().sum(),
    'unique': df.nunique(dropna=True),
    'dtype': df.dtypes.astype(str),
}).sort_values('missing', ascending=False)
quality

In [3]:
print('Exact duplicates:', df.duplicated().sum())
print('Rows with missing values:', df.isna().any(axis=1).sum())
print('Duplicate kos-name rows:', df.iloc[:, 1].duplicated().sum())

In [4]:
staging = pd.read_csv('../data/staging/survey_clean_v1.csv')
staging[['nama_kost', 'kecamatan', 'desa_kelurahan', 'luas_kamar_m2', 'kategori_lantai', 'keamanan_kos', 'harga_sewa', 'periode_sewa', 'harga_bulanan_norm', 'accepted_for_analysis', 'model_ready']].head(20)

In [5]:
staging['harga_bulanan_norm'].describe()

## Catatan Profiling (928 Baris)

- **Completeness:** 37 sel kosong pada raw berhasil dinormalisasi menjadi 0 sel kosong pada staging (`survey_clean_v1.csv`).
- **Validity:** 42 ekspresi perkalian luas kamar (`4x4`, `3,5x5`) dihitung otomatis; typo luas `2000` -> `200` m² dan jarak `300010` -> `3000` m dikoreksi; lantai `'bawah tanah'` -> `1` dan lantai `> 5` -> `'4 - 5'`.
- **Consistency:** Seluruh jarak menggunakan satuan **meter (`_m`)** disertai 7 kolom `klasifikasi_jarak_*`; kategori keamanan `'CCTV Saja'` dan `'CCTV'` disatukan menjadi `'CCTV'`.
- **Uniqueness:** 0 exact duplicate; 42 baris ditandai sebagai `duplicate_logis` tanpa menghapus raw data.
- **Kesiapan Data:** 923 baris lolos analisis (`fact_sewa_kos`) dan 833 baris berstatus `model_ready = 1`.